# 03 — Multi-OU Model Calibration

## Objective

The empirical analysis indicates that SPY volatility contains both short-lived and highly persistent components. This notebook translates that evidence into a parsimonious two-factor stochastic volatility model.

The latent adjusted log-volatility state is represented as

$$
X_t
=
X_t^{(f)}
+
X_t^{(s)},
$$

where each component follows an independent Ornstein–Uhlenbeck process,

$$
dX_t^{(i)}
=
-\lambda_i X_t^{(i)}\,dt
+
\sigma_i\,dW_t^{(i)}.
$$

For a stationary OU factor,

$$
w_i
=
\operatorname{Var}(X_t^{(i)})
=
\frac{\sigma_i^2}{2\lambda_i},
$$

and therefore

$$
\sigma_i
=
\sqrt{2\lambda_iw_i}.
$$

The resulting covariance kernel is

$$
K(\tau)
=
\sum_i w_i e^{-\lambda_i\tau}.
$$

The observed 10-minute adjusted log realised variance is modelled as

$$
Y_t
=
X_t^{(f)}
+
X_t^{(s)}
+
\epsilon_t,
\qquad
\epsilon_t \sim N(0,R).
$$

To preserve a genuine out-of-sample forecasting experiment, the dataset is divided chronologically into:

- **training:** observations before 2017;
- **validation:** 2017–2018;
- **test:** 2019 onward.

All deterministic seasonality and OU parameters are estimated using training data only. The validation and test periods do not influence structural calibration.

This notebook compares Single-OU and Multi-OU covariance specifications, estimates the latent factor parameters and observation-noise variance, and produces the calibrated state-space model used in the forecasting experiment.

In [1]:
import sys

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.append("..")

from implementation.covariance_estimation import estimate_covariance_data

%load_ext autoreload
%autoreload 2

In [2]:
file_path = "../data/SPY_10min_rv.csv"

df = pd.read_csv(file_path)

df["interval_start"] = pd.to_datetime(df["interval_start"])
df["interval_end"] = pd.to_datetime(df["interval_end"])

data = df[
    df["valid"]
    & df["log_realised_variance"].notna()
].copy()

data = data.sort_values("interval_start").reset_index(drop=True)

print("Observations:", len(data))
print("Start:", data["interval_start"].min())
print("End:", data["interval_start"].max())

Observations: 193120
Start: 2002-12-30 09:30:00
End: 2026-09-04 15:30:00


In [3]:
train_end = pd.Timestamp("2017-01-01")
validation_end = pd.Timestamp("2019-01-01")

train = data[
    data["interval_start"] < train_end
].copy()

validation = data[
    (data["interval_start"] >= train_end)
    & (data["interval_start"] < validation_end)
].copy()

test = data[
    data["interval_start"] >= validation_end
].copy()

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Train: 113209
Validation: 16301
Test: 63610


In [4]:
train["clock_time"] = train["interval_start"].dt.strftime("%H:%M")
validation["clock_time"] = validation["interval_start"].dt.strftime("%H:%M")
test["clock_time"] = test["interval_start"].dt.strftime("%H:%M")

seasonal_mean = train.groupby("clock_time")["log_realised_variance"].mean()

train["seasonal_mean"] = train["clock_time"].map(seasonal_mean)
validation["seasonal_mean"] = validation["clock_time"].map(seasonal_mean)
test["seasonal_mean"] = test["clock_time"].map(seasonal_mean)

train["adjusted_log_rv"] = train["log_realised_variance"] - train["seasonal_mean"]
validation["adjusted_log_rv"] = validation["log_realised_variance"] - validation["seasonal_mean"]
test["adjusted_log_rv"] = test["log_realised_variance"] - test["seasonal_mean"]

print("Train mean:", train["adjusted_log_rv"].mean())
print("Validation mean:", validation["adjusted_log_rv"].mean())
print("Test mean:", test["adjusted_log_rv"].mean())

Train mean: -9.640520118784848e-17
Validation mean: -0.8935095454798918
Test mean: -0.2592990502938747


In [5]:
covariance_data = estimate_covariance_data(train)

print("Intraday points:", (covariance_data["source"] == "Intraday").sum())
print("Interday points:", (covariance_data["source"] == "Interday").sum())
print("Total points:", len(covariance_data))

covariance_data.head()

Intraday points: 36
Interday points: 30
Total points: 66


,effective_hours,covariance,pairs,source
0,0.166667,1.141636,97004,Intraday
1,0.333333,1.114695,90479,Intraday
2,0.500000,1.096276,88100,Intraday
3,0.666667,1.082398,85584,Intraday
4,0.833333,1.066153,82891,Intraday


In [6]:
fig = px.scatter(
    covariance_data,
    x="effective_hours",
    y="covariance",
    color="source",
    log_x=True,
    title="Training Volatility Memory"
)

fig.update_layout(
    xaxis_title="Effective Market Time (Hours)",
    yaxis_title="Covariance"
)

fig.show()

In [7]:
from scipy.optimize import curve_fit


def single_ou_covariance(t, weight, mean_reversion):
    return weight * np.exp(-mean_reversion * t)


lags = covariance_data["effective_hours"].to_numpy()
covariance = covariance_data["covariance"].to_numpy()

single_parameters, _ = curve_fit(
    single_ou_covariance,
    lags,
    covariance,
    p0=[covariance[0], 0.01],
    bounds=(0, np.inf),
    maxfev=100000
)

single_weight = single_parameters[0]
single_mean_reversion = single_parameters[1]

single_fit = single_ou_covariance(lags, single_weight, single_mean_reversion)
single_mse = np.mean((covariance - single_fit) ** 2)

print("SINGLE OU")
print("Weight:", single_weight)
print("Mean reversion:", single_mean_reversion)
print("Timescale:", 1 / single_mean_reversion, "effective hours")
print("MSE:", single_mse)

SINGLE OU
Weight: 0.9938549587879484
Mean reversion: 0.00282856294511316
Timescale: 353.5364138626209 effective hours
MSE: 0.0022328381665724


In [8]:
def two_ou_covariance(t, weight_1, mean_reversion_1, weight_2, mean_reversion_2):
    return (
        weight_1 * np.exp(-mean_reversion_1 * t)
        + weight_2 * np.exp(-mean_reversion_2 * t)
    )


two_parameters, _ = curve_fit(
    two_ou_covariance,
    lags,
    covariance,
    p0=[0.3, 0.3, 0.9, 0.0025],
    bounds=(0, np.inf),
    maxfev=100000
)

weight_1 = two_parameters[0]
mean_reversion_1 = two_parameters[1]
weight_2 = two_parameters[2]
mean_reversion_2 = two_parameters[3]

# Keep factor 1 as the fast factor.
if mean_reversion_1 < mean_reversion_2:
    weight_1, weight_2 = weight_2, weight_1
    mean_reversion_1, mean_reversion_2 = mean_reversion_2, mean_reversion_1

two_fit = two_ou_covariance(
    lags,
    weight_1,
    mean_reversion_1,
    weight_2,
    mean_reversion_2
)

two_mse = np.mean((covariance - two_fit) ** 2)

print("TWO OU")
print()
print("Fast factor")
print("Weight:", weight_1)
print("Mean reversion:", mean_reversion_1)
print("Timescale:", 1 / mean_reversion_1, "effective hours")
print()
print("Slow factor")
print("Weight:", weight_2)
print("Mean reversion:", mean_reversion_2)
print("Timescale:", 1 / mean_reversion_2, "effective hours")
print()
print("MSE:", two_mse)
print("MSE reduction vs single OU:", 1 - two_mse / single_mse)

TWO OU

Fast factor
Weight: 0.2121630007593552
Mean reversion: 0.2780002492326782
Timescale: 3.597119077267549 effective hours

Slow factor
Weight: 0.9022287782691211
Mean reversion: 0.002012113978966434
Timescale: 496.9897383813574 effective hours

MSE: 0.0003934165246832404
MSE reduction vs single OU: 0.8238042816658008


In [9]:
plot_lags = np.logspace(
    np.log10(lags.min()),
    np.log10(lags.max()),
    500
)

fig = px.scatter(
    covariance_data,
    x="effective_hours",
    y="covariance",
    color="source",
    log_x=True,
    title="Training Memory — Single OU vs Two OU"
)

fig.add_scatter(
    x=plot_lags,
    y=single_ou_covariance(
        plot_lags,
        single_weight,
        single_mean_reversion
    ),
    mode="lines",
    name="Single OU"
)

fig.add_scatter(
    x=plot_lags,
    y=two_ou_covariance(
        plot_lags,
        weight_1,
        mean_reversion_1,
        weight_2,
        mean_reversion_2
    ),
    mode="lines",
    name="Two OU"
)

fig.update_layout(
    xaxis_title="Effective Market Time (Hours)",
    yaxis_title="Covariance"
)

fig.show()

In [10]:
observed_variance = np.var(train["adjusted_log_rv"], ddof=0)
latent_variance = weight_1 + weight_2
observation_variance = observed_variance - latent_variance

print("Observed variance:", observed_variance)
print("Latent variance:", latent_variance)
print("Observation variance:", observation_variance)
print("Observation noise share:", observation_variance / observed_variance)

Observed variance: 1.5054410196640058
Latent variance: 1.1143917790284763
Observation variance: 0.3910492406355295
Observation noise share: 0.25975726416887884


In [11]:
calibration = pd.DataFrame({
    "factor": ["fast", "slow"],
    "weight": [weight_1, weight_2],
    "mean_reversion": [mean_reversion_1, mean_reversion_2],
    "timescale_hours": [1 / mean_reversion_1, 1 / mean_reversion_2],
    "sigma": [
        np.sqrt(2 * mean_reversion_1 * weight_1),
        np.sqrt(2 * mean_reversion_2 * weight_2)
    ]
})

calibration

,factor,weight,mean_reversion,timescale_hours,sigma
0,fast,0.212163,0.278000,3.597119,0.343457
1,slow,0.902229,0.002012,496.989738,0.060256


In [12]:
print("Single OU MSE:", single_mse)
print("Two OU MSE:", two_mse)
print("MSE reduction:", 1 - two_mse / single_mse)
print("Observation variance:", observation_variance)
print("Observation noise share:", observation_variance / observed_variance)

Single OU MSE: 0.0022328381665724
Two OU MSE: 0.0003934165246832404
MSE reduction: 0.8238042816658008
Observation variance: 0.3910492406355295
Observation noise share: 0.25975726416887884


In [14]:
import json


parameters = {
    "single_ou": {
        "weight": single_weight,
        "mean_reversion": single_mean_reversion,
        "observation_variance": observed_variance - single_weight
    },
    "multi_ou": {
        "weights": [weight_1, weight_2],
        "mean_reversions": [mean_reversion_1, mean_reversion_2],
        "observation_variance": observation_variance
    }
}

with open("../data/ou_parameters.json", "w") as file:
    json.dump(parameters, file, indent=4)
    
seasonal_mean.to_csv("../data/training_seasonal_profile.csv")



## Conclusion

The multi-timescale structure identified in the exploratory analysis remains clearly visible when calibration is restricted to the pre-2017 training sample.

The fitted two-factor covariance model is

$$
K(\tau)
=
w_f e^{-\lambda_f\tau}
+
w_s e^{-\lambda_s\tau},
$$

with characteristic timescales

$$
T_f
=
\frac{1}{\lambda_f}
\approx
3.6 \text{ effective hours},
$$

and

$$
T_s
=
\frac{1}{\lambda_s}
\approx
497 \text{ effective hours}.
$$

The corresponding stationary covariance weights are approximately

$$
w_f \approx 0.212,
\qquad
w_s \approx 0.902.
$$

Relative to a Single-OU specification fitted to exactly the same empirical covariance observations, the two-factor model reduces covariance-fitting MSE by approximately

$$
\boxed{82\%}.
$$

This provides training-sample evidence that a single mean-reversion rate is insufficient to describe the observed persistence structure.

The variance of adjusted 10-minute log realised variance can be decomposed as

$$
\operatorname{Var}(Y_t)
=
w_f+w_s+R.
$$

Using the fitted latent covariance gives

$$
R\approx0.391,
$$

so approximately $26\%$ of observed adjusted log-RV variance is attributed to observation noise under the fitted state-space specification.

The resulting model is therefore fully specified by

$$
\left(
\lambda_f,
\lambda_s,
w_f,
w_s,
R
\right),
$$

with diffusion coefficients determined by

$$
\sigma_i
=
\sqrt{2\lambda_iw_i}.
$$

The overnight market clock is deliberately not inferred from the covariance curve. Exploratory calibration showed that very slow relaxation and closed-market clock speed are difficult to identify separately from second-order dependence alone. Instead, the overnight clock is treated as a forecasting hyperparameter and selected exclusively using the 2017–2018 validation period.

The post-2019 sample remains untouched.

The next notebook uses exact OU transitions and Kalman filtering to estimate the latent volatility state sequentially, select the overnight clock on validation data, and compare Single-OU and Multi-OU realised-variance forecasts on the final out-of-sample test period.